In [17]:
import pandas as pd
import numpy as np
import datatable as dt
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
from statistics import mean
from sklearnex import patch_sklearn
patch_sklearn()

Intel(R) Extension for Scikit-learn* enabled (https://github.com/intel/scikit-learn-intelex)


In [2]:
phenotype_df = pd.read_csv('/home/student2024/Desktop/Project_Depression/combined_dataset_phenotypes_corrected.csv')
phenotype_df.head()

,ID,Sex,Age,Depression,Study,depr_one_hot
0,GSM3562834_3999840035_R01C01,0,73,Control,1,0
1,GSM3562835_3999840035_R01C02,1,32,Case,1,1
2,GSM3562836_3999840035_R02C01,0,51,Case,1,1
3,GSM3562837_3999840035_R02C02,1,30,Case,1,1
4,GSM3562838_3999840035_R03C01,0,53,Case,1,1


In [3]:
beta = dt.fread('/home/student2024/Desktop/Project_Depression/Combined_beta_non_adj.csv')
beta = beta.to_pandas()
#beta = beta.rename(columns={'C0':'ID'})
beta = beta.T
beta.columns = beta.iloc[0]
beta = beta[1:]

beta.head()

C0,cg00000957,cg00001583,cg00002028,cg00002719,cg00002837,cg00003202,cg00003287,cg00007036,cg00007898,cg00008004,...,cg27612019,cg27614432,cg27634071,cg27634744,cg27648216,cg27656573,cg27657363,cg27657537,cg27662611,cg27665648
GSM3562834_3999840035_R01C01,0.928545,0.039386,0.018139,0.084993,0.24706,0.00972,0.103944,0.98261,0.021978,0.100398,...,0.569339,0.031638,0.03021,0.925235,0.925051,0.929117,0.817449,0.097799,0.038094,0.793473
GSM3562835_3999840035_R01C02,0.934262,0.021361,0.023693,0.066703,0.106444,0.009691,0.117379,0.985386,0.019556,0.121006,...,0.62388,0.040766,0.044453,0.908123,0.92057,0.939773,0.897334,0.083995,0.034091,0.791445
GSM3562836_3999840035_R02C01,0.958892,0.024772,0.019229,0.062244,0.196466,0.009898,0.058603,0.986799,0.019803,0.087895,...,0.653626,0.03575,0.024789,0.896234,0.947952,0.933373,0.892951,0.124723,0.041066,0.824575
GSM3562837_3999840035_R02C02,0.93864,0.032863,0.021346,0.101691,0.203318,0.011023,0.08326,0.981915,0.022004,0.08598,...,0.541783,0.038687,0.036991,0.886639,0.934497,0.925345,0.912454,0.090865,0.031113,0.802558
GSM3562838_3999840035_R03C01,0.948645,0.04015,0.020483,0.104516,0.327028,0.01168,0.121258,0.982077,0.020576,0.129376,...,0.57364,0.02748,0.024269,0.876956,0.925348,0.933851,0.889815,0.079895,0.045262,0.884502


In [4]:
beta.index.name = 'ID'
beta.reset_index(inplace=True)
beta.head()

C0,ID,cg00000957,cg00001583,cg00002028,cg00002719,cg00002837,cg00003202,cg00003287,cg00007036,cg00007898,...,cg27612019,cg27614432,cg27634071,cg27634744,cg27648216,cg27656573,cg27657363,cg27657537,cg27662611,cg27665648
0,GSM3562834_3999840035_R01C01,0.928545,0.039386,0.018139,0.084993,0.24706,0.00972,0.103944,0.98261,0.021978,...,0.569339,0.031638,0.03021,0.925235,0.925051,0.929117,0.817449,0.097799,0.038094,0.793473
1,GSM3562835_3999840035_R01C02,0.934262,0.021361,0.023693,0.066703,0.106444,0.009691,0.117379,0.985386,0.019556,...,0.62388,0.040766,0.044453,0.908123,0.92057,0.939773,0.897334,0.083995,0.034091,0.791445
2,GSM3562836_3999840035_R02C01,0.958892,0.024772,0.019229,0.062244,0.196466,0.009898,0.058603,0.986799,0.019803,...,0.653626,0.03575,0.024789,0.896234,0.947952,0.933373,0.892951,0.124723,0.041066,0.824575
3,GSM3562837_3999840035_R02C02,0.93864,0.032863,0.021346,0.101691,0.203318,0.011023,0.08326,0.981915,0.022004,...,0.541783,0.038687,0.036991,0.886639,0.934497,0.925345,0.912454,0.090865,0.031113,0.802558
4,GSM3562838_3999840035_R03C01,0.948645,0.04015,0.020483,0.104516,0.327028,0.01168,0.121258,0.982077,0.020576,...,0.57364,0.02748,0.024269,0.876956,0.925348,0.933851,0.889815,0.079895,0.045262,0.884502


In [6]:
combined_df = beta.merge(phenotype_df, how='outer', on='ID')
combined_df.head()

,ID,cg00000957,cg00001583,cg00002028,cg00002719,cg00002837,cg00003202,cg00003287,cg00007036,cg00007898,...,cg27656573,cg27657363,cg27657537,cg27662611,cg27665648,Sex,Age,Depression,Study,depr_one_hot
0,GSM3562834_3999840035_R01C01,0.928545,0.039386,0.018139,0.084993,0.24706,0.00972,0.103944,0.98261,0.021978,...,0.929117,0.817449,0.097799,0.038094,0.793473,0,73,Control,1,0
1,GSM3562835_3999840035_R01C02,0.934262,0.021361,0.023693,0.066703,0.106444,0.009691,0.117379,0.985386,0.019556,...,0.939773,0.897334,0.083995,0.034091,0.791445,1,32,Case,1,1
2,GSM3562836_3999840035_R02C01,0.958892,0.024772,0.019229,0.062244,0.196466,0.009898,0.058603,0.986799,0.019803,...,0.933373,0.892951,0.124723,0.041066,0.824575,0,51,Case,1,1
3,GSM3562837_3999840035_R02C02,0.93864,0.032863,0.021346,0.101691,0.203318,0.011023,0.08326,0.981915,0.022004,...,0.925345,0.912454,0.090865,0.031113,0.802558,1,30,Case,1,1
4,GSM3562838_3999840035_R03C01,0.948645,0.04015,0.020483,0.104516,0.327028,0.01168,0.121258,0.982077,0.020576,...,0.933851,0.889815,0.079895,0.045262,0.884502,0,53,Case,1,1


In [7]:
combined_df.shape

(1630, 391216)

In [8]:
combined_df = combined_df.drop(columns=['ID','Depression'])

In [9]:
lr = LogisticRegression(max_iter = 10000)
rf = RandomForestClassifier()

In [10]:
sgkf = StratifiedGroupKFold(shuffle = True)
lr_accu = []
lr_auc = []
rf_mean_accu = []
rf_mean_auc = []

In [11]:
x = combined_df.drop(columns=['Study','depr_one_hot'])
y = combined_df['depr_one_hot']
groups = combined_df['Study']

In [12]:
for train_index, test_index in sgkf.split(x, y, groups):
    x_train_fold, x_test_fold = x.iloc[train_index], x.iloc[test_index]
    y_train_fold, y_test_fold = y[train_index], y[test_index]
    lr.fit(x_train_fold, y_train_fold)
    lr_pred = lr.predict(x_test_fold)
    lr_accu.append(accuracy_score(y_test_fold, lr_pred))
    lr_auc.append(roc_auc_score(y_test_fold, lr_pred))
    rf_accu = 0
    rf_auc = 0
    for i in range(20):
        rf.fit(x_train_fold, y_train_fold)
        rf_pred = rf.predict(x_test_fold)
        rf_accu += accuracy_score(y_test_fold, rf_pred)
        rf_auc += roc_auc_score(y_test_fold, rf_pred)
    rf_mean_accu.append(rf_accu/20)
    rf_mean_auc.append(rf_auc/20)

In [13]:
print('Random Forest: \n\tMean Acuracy: ', mean(rf_mean_accu)*100, '% \n\t Mean ROC AUC: ', mean(rf_mean_auc)*100, '% \n')
print('Logistic Regression: \n\tMeanAccuracy: ', mean(lr_accu)*100, '% \n\t Mean ROC AUC: ', mean(lr_auc)*100, '%')

Random Forest: 
	Mean Acuracy:  61.227143928409966 % 
	 Mean ROC AUC:  50.841332396228836 % 

Logistic Regression: 
	MeanAccuracy:  65.49198540972235 % 
	 Mean ROC AUC:  49.66455881805419 %


Random Forest: 
	Mean Acuracy:  61.227143928409966 % 
	 Mean ROC AUC:  50.841332396228836 % 

Logistic Regression: 
	MeanAccuracy:  65.49198540972235 % 
	 Mean ROC AUC:  49.66455881805419 %

In [14]:
print('RF Accuracy: ',rf_mean_accu,'\nROC AUC: ',rf_mean_auc)
print('LR Accuracy: ', lr_accu,'\n ROC AUC: ',lr_auc)

RF Accuracy:  [0.8518348623853214, 0.5209876543209876, 0.7982638888888887, 0.41373390557939915, 0.4765368852459015] 
ROC AUC:  [0.49919354838709673, 0.5501434948979592, 0.49978260869565216, 0.4920705521472393, 0.5008764156834945]
LR Accuracy:  [0.8532110091743119, 0.4567901234567901, 0.7916666666666666, 0.6995708154506438, 0.4733606557377049] 
 ROC AUC:  [0.5, 0.491390306122449, 0.4956521739130435, 0.5, 0.4961854608672169]


RF Accuracy:  [0.8518348623853214, 0.5209876543209876, 0.7982638888888887, 0.41373390557939915, 0.4765368852459015] 
ROC AUC:  [0.49919354838709673, 0.5501434948979592, 0.49978260869565216, 0.4920705521472393, 0.5008764156834945]
LR Accuracy:  [0.8532110091743119, 0.4567901234567901, 0.7916666666666666, 0.6995708154506438, 0.4733606557377049] 
 ROC AUC:  [0.5, 0.491390306122449, 0.4956521739130435, 0.5, 0.4961854608672169]

In [18]:
sss = StratifiedShuffleSplit()
lr_accu = []
lr_auc = []
rf_mean_accu = []
rf_mean_auc = []
for train_index, test_index in sss.split(x, y, groups):
    x_train_fold, x_test_fold = x.iloc[train_index], x.iloc[test_index]
    y_train_fold, y_test_fold = y[train_index], y[test_index]
    lr.fit(x_train_fold, y_train_fold)
    lr_pred = lr.predict(x_test_fold)
    lr_accu.append(accuracy_score(y_test_fold, lr_pred))
    lr_auc.append(roc_auc_score(y_test_fold, lr_pred))
    rf_accu = 0
    rf_auc = 0
    for i in range(20):
        rf.fit(x_train_fold, y_train_fold)
        rf_pred = rf.predict(x_test_fold)
        rf_accu += accuracy_score(y_test_fold, rf_pred)
        rf_auc += roc_auc_score(y_test_fold, rf_pred)
    rf_mean_accu.append(rf_accu/20)
    rf_mean_auc.append(rf_auc/20)

In [19]:
print('Random Forest: \n\tMean Acuracy: ', mean(rf_mean_accu)*100, '% \n\t Mean ROC AUC: ', mean(rf_mean_auc)*100, '% \n')
print('Logistic Regression: \n\tMeanAccuracy: ', mean(lr_accu)*100, '% \n\t Mean ROC AUC: ', mean(lr_auc)*100, '%')

Random Forest: 
	Mean Acuracy:  66.12883435582823 % 
	 Mean ROC AUC:  55.71582491582492 % 

Logistic Regression: 
	MeanAccuracy:  66.93251533742331 % 
	 Mean ROC AUC:  58.71801346801347 %


Random Forest: 
	Mean Acuracy:  66.12883435582823 % 
	 Mean ROC AUC:  55.71582491582492 % 

Logistic Regression: 
	MeanAccuracy:  66.93251533742331 % 
	 Mean ROC AUC:  58.71801346801347 %

In [20]:
print('RF Accuracy: ',rf_mean_accu,'\nROC AUC: ',rf_mean_auc)
print('LR Accuracy: ', lr_accu,'\n ROC AUC: ',lr_auc)

RF Accuracy:  [0.6404907975460123, 0.6233128834355827, 0.6659509202453987, 0.6892638036809817, 0.638957055214724, 0.6506134969325155, 0.6763803680981595, 0.6865030674846627, 0.675766871165644, 0.665644171779141] 
ROC AUC:  [0.5286153198653198, 0.5261363636363637, 0.5505050505050505, 0.5636363636363637, 0.5332575757575758, 0.558560606060606, 0.5916119528619529, 0.572483164983165, 0.5793265993265992, 0.567449494949495]
LR Accuracy:  [0.6441717791411042, 0.6441717791411042, 0.6625766871165644, 0.7239263803680982, 0.656441717791411, 0.6625766871165644, 0.6748466257668712, 0.6932515337423313, 0.656441717791411, 0.6748466257668712] 
 ROC AUC:  [0.561952861952862, 0.5753367003367004, 0.5713804713804714, 0.6355218855218855, 0.5712121212121212, 0.5981481481481481, 0.6207912457912458, 0.5856060606060606, 0.5667508417508418, 0.5851010101010101]


RF Accuracy:  [0.6404907975460123, 0.6233128834355827, 0.6659509202453987, 0.6892638036809817, 0.638957055214724, 0.6506134969325155, 0.6763803680981595, 0.6865030674846627, 0.675766871165644, 0.665644171779141] 
ROC AUC:  [0.5286153198653198, 0.5261363636363637, 0.5505050505050505, 0.5636363636363637, 0.5332575757575758, 0.558560606060606, 0.5916119528619529, 0.572483164983165, 0.5793265993265992, 0.567449494949495]
LR Accuracy:  [0.6441717791411042, 0.6441717791411042, 0.6625766871165644, 0.7239263803680982, 0.656441717791411, 0.6625766871165644, 0.6748466257668712, 0.6932515337423313, 0.656441717791411, 0.6748466257668712] 
 ROC AUC:  [0.561952861952862, 0.5753367003367004, 0.5713804713804714, 0.6355218855218855, 0.5712121212121212, 0.5981481481481481, 0.6207912457912458, 0.5856060606060606, 0.5667508417508418, 0.5851010101010101]

In [25]:
from sklearn.model_selection import StratifiedKFold
skf = StratifiedKFold()
lr_accu = []
lr_auc = []
rf_mean_accu = []
rf_mean_auc = []
for train_index, test_index in skf.split(x, y, groups):
    x_train_fold, x_test_fold = x.iloc[train_index], x.iloc[test_index]
    y_train_fold, y_test_fold = y[train_index], y[test_index]
    lr.fit(x_train_fold, y_train_fold)
    lr_pred = lr.predict(x_test_fold)
    lr_accu.append(accuracy_score(y_test_fold, lr_pred))
    lr_auc.append(roc_auc_score(y_test_fold, lr_pred))
    rf_accu = 0
    rf_auc = 0
    for i in range(20):
        rf.fit(x_train_fold, y_train_fold)
        rf_pred = rf.predict(x_test_fold)
        rf_accu += accuracy_score(y_test_fold, rf_pred)
        rf_auc += roc_auc_score(y_test_fold, rf_pred)
    rf_mean_accu.append(rf_accu/20)
    rf_mean_auc.append(rf_auc/20)

In [26]:
print('Random Forest: \n\tMean Acuracy: ', mean(rf_mean_accu)*100, '% \n\t Mean ROC AUC: ', mean(rf_mean_auc)*100, '% \n')
print('Logistic Regression: \n\tMeanAccuracy: ', mean(lr_accu)*100, '% \n\t Mean ROC AUC: ', mean(lr_auc)*100, '%')

Random Forest: 
	Mean Acuracy:  63.43251533742331 % 
	 Mean ROC AUC:  54.825957257149916 % 

Logistic Regression: 
	MeanAccuracy:  62.45398773006134 % 
	 Mean ROC AUC:  54.435694869181106 %


Random Forest: 
	Mean Acuracy:  63.43251533742331 % 
	 Mean ROC AUC:  54.825957257149916 % 

Logistic Regression: 
	MeanAccuracy:  62.45398773006134 % 
	 Mean ROC AUC:  54.435694869181106 %

In [27]:
print('RF Accuracy: ',rf_mean_accu,'\nROC AUC: ',rf_mean_auc)
print('LR Accuracy: ', lr_accu,'\n ROC AUC: ',lr_auc)

RF Accuracy:  [0.639723926380368, 0.6641104294478527, 0.51840490797546, 0.6634969325153375, 0.6858895705521473] 
ROC AUC:  [0.4917887205387205, 0.503834175084175, 0.6081131357544497, 0.5973555151566398, 0.5402063163235107]
LR Accuracy:  [0.6687116564417178, 0.656441717791411, 0.49693251533742333, 0.6042944785276073, 0.696319018404908] 
 ROC AUC:  [0.509090909090909, 0.515446127946128, 0.5878746882002283, 0.5452373906058429, 0.5641356276159473]


RF Accuracy:  [0.639723926380368, 0.6641104294478527, 0.51840490797546, 0.6634969325153375, 0.6858895705521473] 
ROC AUC:  [0.4917887205387205, 0.503834175084175, 0.6081131357544497, 0.5973555151566398, 0.5402063163235107]
LR Accuracy:  [0.6687116564417178, 0.656441717791411, 0.49693251533742333, 0.6042944785276073, 0.696319018404908] 
 ROC AUC:  [0.509090909090909, 0.515446127946128, 0.5878746882002283, 0.5452373906058429, 0.5641356276159473]

In [28]:
from sklearn.model_selection import GroupShuffleSplit
gss = GroupShuffleSplit()
lr_accu = []
lr_auc = []
rf_mean_accu = []
rf_mean_auc = []
for train_index, test_index in gss.split(x, y, groups):
    x_train_fold, x_test_fold = x.iloc[train_index], x.iloc[test_index]
    y_train_fold, y_test_fold = y[train_index], y[test_index]
    lr.fit(x_train_fold, y_train_fold)
    lr_pred = lr.predict(x_test_fold)
    lr_accu.append(accuracy_score(y_test_fold, lr_pred))
    lr_auc.append(roc_auc_score(y_test_fold, lr_pred))
    rf_accu = 0
    rf_auc = 0
    for i in range(20):
        rf.fit(x_train_fold, y_train_fold)
        rf_pred = rf.predict(x_test_fold)
        rf_accu += accuracy_score(y_test_fold, rf_pred)
        rf_auc += roc_auc_score(y_test_fold, rf_pred)
    rf_mean_accu.append(rf_accu/20)
    rf_mean_auc.append(rf_auc/20)

In [29]:
print('Random Forest: \n\tMean Acuracy: ', mean(rf_mean_accu)*100, '% \n\t Mean ROC AUC: ', mean(rf_mean_auc)*100, '% \n')
print('Logistic Regression: \n\tMeanAccuracy: ', mean(lr_accu)*100, '% \n\t Mean ROC AUC: ', mean(lr_auc)*100, '%')

Random Forest: 
	Mean Acuracy:  59.96263933040481 % 
	 Mean ROC AUC:  57.11488248070532 % 

Logistic Regression: 
	MeanAccuracy:  64.48912837694098 % 
	 Mean ROC AUC:  59.06128996531999 %


Random Forest: 
	Mean Acuracy:  59.96263933040481 % 
	 Mean ROC AUC:  57.11488248070532 % 

Logistic Regression: 
	MeanAccuracy:  64.48912837694098 % 
	 Mean ROC AUC:  59.06128996531999 %

In [30]:
print('RF Accuracy: ',rf_mean_accu,'\nROC AUC: ',rf_mean_auc)
print('LR Accuracy: ', lr_accu,'\n ROC AUC: ',lr_auc)

RF Accuracy:  [0.4369871794871796, 0.7615384615384617, 0.6037766830870279, 0.5213483146067415, 0.6744813278008297] 
ROC AUC:  [0.481712662758728, 0.6037121984369691, 0.6285530138845887, 0.5145984472444954, 0.6271678017104847]
LR Accuracy:  [0.6730769230769231, 0.7725752508361204, 0.6124794745484401, 0.47752808988764045, 0.6887966804979253] 
 ROC AUC:  [0.5184009070765309, 0.622918790349983, 0.6645490485603373, 0.5033091510754741, 0.6438866012036744]


RF Accuracy:  [0.4369871794871796, 0.7615384615384617, 0.6037766830870279, 0.5213483146067415, 0.6744813278008297] 
ROC AUC:  [0.481712662758728, 0.6037121984369691, 0.6285530138845887, 0.5145984472444954, 0.6271678017104847]
LR Accuracy:  [0.6730769230769231, 0.7725752508361204, 0.6124794745484401, 0.47752808988764045, 0.6887966804979253] 
 ROC AUC:  [0.5184009070765309, 0.622918790349983, 0.6645490485603373, 0.5033091510754741, 0.6438866012036744]

In [31]:
import gc
del beta
del phenotype_df
del combined_df
gc.collect()

320

In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit